# LightGBM Runner - v2 Pipeline

3 variants x 5 seeds. Saves predictions to `data/predictions/`.
Optuna search runs once on the `hijri` variant; tuned params reused across variants/seeds.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from pathlib import Path

from src.models.ml.lgbm import LightGBMModel, tune_with_optuna
from src.evaluation.predictions_io import write_predictions, read_predictions
from src.evaluation.regime_eval import evaluate_by_regime
from src.evaluation.bootstrap import block_bootstrap_ci

# Walk up until we find pyproject.toml so notebook works from any cwd.
ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
print(f'ROOT = {ROOT}')

DATA = pd.read_csv(ROOT / 'data' / 'processed' / 'final_training_set_v2.csv',
                   parse_dates=['timestamp']).set_index('timestamp')
DATA.index = DATA.index.tz_convert('UTC') if DATA.index.tz is not None else DATA.index.tz_localize('UTC')
DATA = DATA.dropna(subset=['y_lag_336h', 'y_roll168_mean'])

TRAIN = DATA.loc['2018':'2022']
VAL   = DATA.loc['2023']
TEST  = DATA.loc['2024':'2025-03']
print(f'Train {len(TRAIN):,}  Val {len(VAL):,}  Test {len(TEST):,}')


ROOT = C:\Users\omars\Documents\Ramadan-Aware-Short-Term-Load-Forecasting-Benchmarking-


Train 43,491  Val 8,760  Test 10,944


In [2]:
# 1) Optuna tune ONCE on the hijri variant, seed 42.
best_params = tune_with_optuna(TRAIN, VAL, variant='hijri', n_trials=50, seed=42)
print('Best Optuna params:'); print(best_params)


Best Optuna params:
{'learning_rate': 0.03353646056278733, 'num_leaves': 221, 'max_depth': 11, 'min_child_samples': 51, 'feature_fraction': 0.6412135092571697, 'bagging_fraction': 0.8338997111968103, 'lambda_l1': 0.0008394696905277819, 'lambda_l2': 0.00010076695535621531, 'min_split_gain': 0.10539691941829943}


In [3]:
# 2) Run 3 variants x 5 seeds with the tuned params.
VARIANTS = ['nohijri', 'hijri', 'hijri_plusB']
SEEDS = [42, 43, 44, 45, 46]
for variant in VARIANTS:
    for seed in SEEDS:
        model = LightGBMModel(variant=variant, n_estimators=3000, **best_params)
        model.fit(TRAIN, VAL, hijri=(variant != 'nohijri'), seed=seed)
        preds = model.predict(TEST)
        path = write_predictions(preds, model='lgbm', variant=variant, context_length=None, seed=seed)
        print(f'wrote {path.name} | rows={len(preds)}')


wrote lgbm__nohijri__seed42.parquet | rows=10944


wrote lgbm__nohijri__seed43.parquet | rows=10944


wrote lgbm__nohijri__seed44.parquet | rows=10944


wrote lgbm__nohijri__seed45.parquet | rows=10944


wrote lgbm__nohijri__seed46.parquet | rows=10944


...

wrote lgbm__hijri_plusB__seed42.parquet | rows=10944


wrote lgbm__hijri_plusB__seed43.parquet | rows=10944


wrote lgbm__hijri_plusB__seed44.parquet | rows=10944


wrote lgbm__hijri_plusB__seed45.parquet | rows=10944


wrote lgbm__hijri_plusB__seed46.parquet | rows=10944


In [4]:
# 3) Regime metrics on median-seed predictions (seed 44 is the middle of 42..46).
results = []
for variant in VARIANTS:
    p = read_predictions(model='lgbm', variant=variant, context_length=None, seed=44)
    tab = evaluate_by_regime(
        p['y_true'].values, p['y_pred'].values,
        regimes=p['regime'], y_train=TRAIN['actual_load'].values, period=168,
    )
    tab.insert(0, 'variant', variant)
    results.append(tab)
print(pd.concat(results).to_string(index=False))


    variant   regime    n         mae        rmse     mape     mase
    nohijri   Normal 7992  889.212729 1430.101446 2.379475 0.536034
    nohijri  Ramadan 1416  897.739233 1320.818394 2.525311 0.541174
    nohijri Heatwave 1536 1693.872366 2391.142583 3.682989 1.021098
    nohijri Compound    0         NaN         NaN      NaN      NaN
      hijri   Normal 7992  873.509026 1366.765526 2.288744 0.526568
      hijri  Ramadan 1416  799.944325 1120.516834 2.213045 0.482221
      hijri Heatwave 1536 1692.960375 2395.732397 3.690565 1.020548
      hijri Compound    0         NaN         NaN      NaN      NaN
hijri_plusB   Normal 7992  873.509026 1366.765526 2.288744 0.526568
hijri_plusB  Ramadan 1416  799.944325 1120.516834 2.213045 0.482221
hijri_plusB Heatwave 1536 1692.960375 2395.732397 3.690565 1.020548
hijri_plusB Compound    0         NaN         NaN      NaN      NaN


In [5]:
# 4) Bootstrap CIs on aggregate MAE per variant (across-regime).
for variant in VARIANTS:
    p = read_predictions(model='lgbm', variant=variant, context_length=None, seed=44)
    abs_err = (p['y_true'] - p['y_pred']).abs().values
    lo, hi = block_bootstrap_ci(abs_err, block_size=24, n_resamples=1000, seed=42)
    print(f'{variant:<14} MAE={abs_err.mean():.2f}  CI95=[{lo:.2f}, {hi:.2f}]')


nohijri        MAE=1003.25  CI95=[910.21, 1108.12]


hijri          MAE=979.00  CI95=[894.41, 1078.71]


hijri_plusB    MAE=979.00  CI95=[894.41, 1078.71]
